# MRI II — Oral Exam: Radial GRE Sequence Assembly

**Time limit: 30 minutes**

All building blocks (RF pulse, gradients, ADC, spoiler, timing) are already
constructed for you below. **Your only task is to assemble the sequence** —
i.e. write the loop that calls `seq.add_block()` in the correct order and
structure for this trajectory. Once correct, the rest of the notebook
(timing check, simulation, reconstruction) will run and produce an image —
that image is your feedback.



In [ ]:
%matplotlib widget

import sys

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch

sys.path.insert(0, "/home/jovyan/mri/src")
sys.path.insert(1, "/home/jovyan/shared-folder/Magnetresonanztomographie-2/mri/src")

from mri.conopt.functional.fourier import nonuniform_fourier_transform_adjoint

In [ ]:
seq_filename = "gre_radial_exam.seq"

fov_x, fov_y = 256e-3, 256e-3
n_x, n_y = 128, 128
R = 1  # fully-sampled radial acquisition -- kept fixed for the exam

flip_angle_deg = 40
tr = 5000e-3
te = 5e-3

torch.set_default_device("cpu")

## System and Sequence Object

`system` holds the scanner hardware limits. `seq` is the container you add
blocks (RF pulses, gradients, delays, ADC windows) to, in chronological order.
Objects passed together to a single `seq.add_block()` call play *simultaneously*.


In [ ]:
system = pp.Opts(
    max_grad=28,
    grad_unit="mT/m",
    max_slew=150,
    slew_unit="T/m/s",
)

seq = pp.Sequence(system)

## Building Blocks (given)

- `rf` — excitation pulse
- `g_ro` — the (unrotated) readout gradient, along x
- `g_pre` — the (unrotated) readout pre-phaser, along x
- `adc` — readout window
- `gx_spoiler`, `gy_spoiler` — spoilers on both axes (needed since the readout
  direction changes every TR)
- `te_delay`, `tr_delay` — computed for a single spoke


In [ ]:
delta_kx = 1 / fov_x
raster_eps = 1e-12

rf = pp.make_sinc_pulse(
    flip_angle=np.deg2rad(flip_angle_deg),
    system=system,
    use="excitation",
)

g_ro = pp.make_trapezoid(
    channel="x",
    flat_area=n_x * delta_kx,
    flat_time=3.2e-3,
    system=system,
)

g_pre = pp.make_trapezoid(
    channel="x",
    area=-g_ro.area / 2,
    system=system,
)

adc = pp.make_adc(
    num_samples=n_x,
    duration=g_ro.flat_time,
    delay=g_ro.rise_time,
    system=system,
)

gx_spoiler = pp.make_trapezoid(
    channel="x",
    area=4 * g_ro.area,
    system=system,
)

gy_spoiler = pp.make_trapezoid(
    channel="y",
    area=4 * g_ro.area,
    system=system,
)

te_delay = (
    te
    - (pp.calc_duration(rf) - pp.calc_rf_center(rf)[0])
    - pp.calc_duration(g_pre)
    - pp.calc_duration(g_ro) / 2
)

tr_delay = (
    tr
    - pp.calc_duration(rf)
    - pp.calc_duration(g_pre)
    - pp.calc_duration(g_ro)
    - pp.calc_duration(gx_spoiler)
    - te_delay
)

te_delay = (
    np.ceil((te_delay - raster_eps) / seq.grad_raster_time) * seq.grad_raster_time
)
tr_delay = (
    np.ceil((tr_delay - raster_eps) / seq.grad_raster_time) * seq.grad_raster_time
)

assert np.all(te_delay >= 0)
assert np.all(tr_delay >= 0)

## Task: Assemble the Radial GRE Sequence

In radial imaging, each TR acquires one **spoke** through the center of
k-space at a different angle, instead of a parallel line. The spoke angle for
spoke $i$ is:

$$\theta_i = \frac{i \cdot \pi}{n_y}, \qquad i = 0, \dots, n_y - 1$$

For each angle, the pre-phaser and readout are rotated by projecting
`g_pre.area` / `g_ro.area` onto x and y:

$$g_{x,\text{pre}} = \cos(\theta)\, g_\text{pre}, \quad g_{y,\text{pre}} = \sin(\theta)\, g_\text{pre}$$
$$g_{x,\text{ro}} = \cos(\theta)\, g_\text{ro}, \quad g_{y,\text{ro}} = \sin(\theta)\, g_\text{ro}$$

**For each spoke:**
1. Add the RF pulse.
2. Build `gx_pre`/`gy_pre` on channels `"x"`/`"y"` with the rotated areas above
   and `duration=pp.calc_duration(g_pre)`.
3. Add `gx_pre` and `gy_pre` together.
4. Add the TE delay.
5. Build `gx_ro`/`gy_ro` the same way (rotated `g_ro`, `duration=pp.calc_duration(g_ro)`).
6. Add `gx_ro`, `gy_ro` and `adc` together.
7. Add `gx_spoiler` and `gy_spoiler` together.
8. Add the TR delay.

**Hint:** `angles = np.linspace(0, np.pi, num=n_y, endpoint=False)`.


In [ ]:
raise NotImplementedError


## Verification, Simulation, Reconstruction (given)

In [ ]:
ok, error_report = seq.check_timing()

if ok:
    print("Timing check passed successfully")
else:
    print("Timing check failed. Error listing follows:")
    [print(e) for e in error_report]

plt.figure()
seq.plot()

seq.write(seq_filename, v141_compat=True)

In [ ]:
phantom = mr0.util.load_phantom(size=(n_x, n_y))

plt.figure()
phantom.plot()

mr0_seq = mr0.Sequence.import_file(seq_filename)

data = phantom.build()
graph = mr0.compute_graph(mr0_seq, data)
signal = mr0.execute_graph(graph, mr0_seq, data, print_progress=True)

## Reconstruction (given)

Radial samples don't lie on a Cartesian grid, so a plain FFT cannot be used.
This uses the course's Non-Uniform FFT (NUFFT) adjoint, with the true
k-space trajectory read directly from the assembled sequence via
`seq.calculate_kspace()`.


In [ ]:
data = signal.reshape(1, -1)  # single receive channel

kspace = seq.calculate_kspace()
k_traj_adc = kspace[0][:2, :]  # keep kx, ky only
k_traj_adc = k_traj_adc / np.sqrt(np.sum(k_traj_adc**2, axis=0)).max() * 0.5
k_traj_adc = torch.tensor(k_traj_adc, dtype=torch.float32)

k_filter_eps = 1e-6
k_filter = (
    torch.linalg.norm(
        k_traj_adc,
        ord=2,
        dim=0,
        keepdims=True,
    )
    + k_filter_eps
)

image = nonuniform_fourier_transform_adjoint(
    k_traj_adc,
    k_filter * data,
    (n_x, n_y),
)[0, ..., 0]

plt.figure()
plt.imshow(torch.abs(image).numpy().T, cmap="gray")
plt.title("Reconstructed Radial GRE")
plt.show()